# Task 1: Vehicle Market Valuation Prediction Engine
This notebook demonstrates a rigorous data science workflow to analyze and predict vehicle resale values using an optimized Gradient Boosting approach.


In [ ]:
# STEP 1: Libraries Ingestion & Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import joblib

# Aesthetics configuration for plots
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


In [ ]:
# STEP 2: Data Loading & Verification
df = pd.read_csv('car_data.csv')

print("--- Initial Inspection ---")
print(f"Dataset Dimensions: {df.shape}")
print("\nFirst 5 Records:")
print(df.head())


In [ ]:
# STEP 3: Strict Data Cleaning & Handling NaNs
print("--- Checking for Missing Values (NaNs) ---")
missing_data = df.isnull().sum()
print(missing_data)

# Handling NaNs logically if they exist
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print("\nMissing values after strict cleaning treatment:")
print(df.isnull().sum())


In [ ]:
# STEP 4: Feature Engineering & Preprocessing
# Creating a relative 'Car_Age' metric instead of absolute 'Year' to map depreciation
current_year = 2026
df['Car_Age'] = current_year - df['Year']

# Dropping unnecessary reference columns to prevent target leakage or low variance noise
df = df.drop(columns=['Year', 'Car_Name'])

print("Processed Features Summary Statistics:")
print(df.describe(include='all'))


In [ ]:
# STEP 5: Exploratory Data Analysis (EDA)
plt.figure(figsize=(8, 5))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='RdYlBu', fmt='.2f')
plt.title('Correlation Matrix of Numeric Features')
plt.show()


In [ ]:
# STEP 6: Splitting Features & Targets
X = df.drop(columns=['Selling_Price'])
y = df['Selling_Price']

# Stratified partition into operational validation pairs
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# STEP 7: Transformation Pipeline Setup
categorical_features = ['Fuel_Type', 'Selling_type', 'Transmission']
numerical_features = ['Present_Price', 'Driven_kms', 'Owner', 'Car_Age']

# Encoding maps via ColumnTransformer workflow
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

# Unified Ensemble Pipeline Design
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, random_state=42))
])

# Training the predictive framework
model_pipeline.fit(X_train, y_train)
print("[SUCCESS] Production optimization framework trained successfully!")


In [ ]:
# STEP 8: Evaluation Profiles
y_pred = model_pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("--- Model Performance Metrics ---")
print(f"R-squared (Variance Accounted For): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")


In [ ]:
# STEP 9: Serialization & Export
joblib.dump(model_pipeline, 'car_price_predictor.pkl')
print("[DEPLOY] Model serialized into 'car_price_predictor.pkl' file.")
